# Notebook 10 — Score d'alerte boutiques (SB-Z)

**Objectif** : produire un score d'alerte actionnable pour les boutiques, en croisant :
- **Signal OCG** : archetypes dominants en OCG aujourd'hui → seront forts en TCG dans ~4 mois
- **Signal views_week** : cartes dont l'intérêt explose sur YGOPRODeck → signal d'achat avant les tournois

**Formule** : `alert_score = meta_score_ocg × log(1 + avg_views_week_cartes_core)`

**Filtres appliqués** : staples format (>20% des decks TCG) et cartes bannies TCG exclues.

**Prérequis** : avoir lancé `python scripts/fetch_ocg_decks.py`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime, date
from dateutil.relativedelta import relativedelta

DB = os.path.join(os.getcwd(), '..', 'data', 'yugioh.db')
conn = sqlite3.connect(DB)

LAG_MOIS = 4
FENETRE_OCG_MOIS = 3

ocg_n = pd.read_sql('SELECT COUNT(*) as n FROM tournament_decks WHERE ocg=1', conn).iloc[0,0]
tcg_n = pd.read_sql('SELECT COUNT(*) as n FROM tournament_decks WHERE ocg=0', conn).iloc[0,0]
print(f'Decklists OCG : {ocg_n}')
print(f'Decklists TCG : {tcg_n}')
print(f'Lag validé    : {LAG_MOIS} mois')

## Part A — Meta score OCG récent (3 derniers mois)

In [ ]:
ocg_raw = pd.read_sql("""
    SELECT strftime('%Y-%m', created) as month, archetype,
           COUNT(*) as n_decks,
           AVG(CASE WHEN placement IS NOT NULL THEN placement END) as avg_placement
    FROM tournament_decks
    WHERE ocg=1 AND archetype IS NOT NULL AND illegal=0
    GROUP BY month, archetype
""", conn)

ocg_raw['month'] = pd.to_datetime(ocg_raw['month'])
cutoff = ocg_raw['month'].max() - pd.DateOffset(months=FENETRE_OCG_MOIS)
ocg_recent = ocg_raw[ocg_raw['month'] >= cutoff].copy()

print(f'Période OCG : {cutoff.strftime("%Y-%m")} → {ocg_raw["month"].max().strftime("%Y-%m")}')

agg = ocg_recent.groupby('archetype').agg(
    n_decks=('n_decks', 'sum'),
    avg_placement=('avg_placement', 'mean')
).reset_index()

total = agg['n_decks'].sum()
agg['share_ocg'] = agg['n_decks'] / total
agg['placement_score'] = 1 / agg['avg_placement'].replace(0, np.nan)
agg['placement_score_norm'] = agg['placement_score'] / agg['placement_score'].max()
agg['meta_score_ocg'] = np.sqrt(agg['share_ocg'] * agg['placement_score_norm'].fillna(agg['share_ocg']))
agg = agg.sort_values('meta_score_ocg', ascending=False)

print()
print('Top 20 archetypes OCG récents :')
print(agg[['archetype', 'n_decks', 'share_ocg', 'meta_score_ocg']].head(20).to_string(index=False))

## Part B — Cartes clés OCG + views_week (staples et bannies exclues)

In [ ]:
# Decks OCG récents
ocg_deck_ids = pd.read_sql("""
    SELECT id, archetype FROM tournament_decks
    WHERE ocg=1 AND archetype IS NOT NULL AND illegal=0
    AND created >= date('now', '-3 months')
""", conn)

if len(ocg_deck_ids) == 0:
    ocg_deck_ids = pd.read_sql("""
        SELECT id, archetype FROM tournament_decks
        WHERE ocg=1 AND archetype IS NOT NULL AND illegal=0
    """, conn)

# Cartes jouées dans ces decks
ids_list = "','".join(ocg_deck_ids['id'].tolist())
ocg_cards = pd.read_sql(f"""
    SELECT dc.deck_id, dc.card_name, dc.zone
    FROM deck_cards dc
    WHERE dc.deck_id IN ('{ids_list}') AND dc.zone = 'main'
""", conn)

ocg_cards = ocg_cards.merge(ocg_deck_ids[['id', 'archetype']], left_on='deck_id', right_on='id')

# Fréquence par carte par archetype
arch_size = ocg_deck_ids.groupby('archetype')['id'].count().rename('n_decks_arch')
card_freq = ocg_cards.groupby(['archetype', 'card_name'])['deck_id'].count().reset_index()
card_freq.columns = ['archetype', 'card_name', 'n_decks_played']
card_freq = card_freq.join(arch_size, on='archetype')
card_freq['frequency'] = card_freq['n_decks_played'] / card_freq['n_decks_arch']
key_cards = card_freq[card_freq['frequency'] >= 0.3].copy()

# Ajouter views_week
views = pd.read_sql("SELECT name, views_week, views FROM cards", conn)
key_cards = key_cards.merge(views, left_on='card_name', right_on='name', how='left')
key_cards['views_week'] = key_cards['views_week'].fillna(0)

# Filtre 1 : staples format (présentes dans >20% des decks TCG)
tcg_total = pd.read_sql("SELECT COUNT(*) as n FROM tournament_decks WHERE ocg=0", conn).iloc[0,0]
staples = pd.read_sql(f"""
    SELECT dc.card_name, COUNT(DISTINCT dc.deck_id) * 1.0 / {tcg_total} as global_freq
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.ocg = 0
    GROUP BY dc.card_name
    HAVING global_freq > 0.20
""", conn)
staple_list = set(staples['card_name'].tolist())

# Filtre 2 : cartes bannies TCG
banned = pd.read_sql("SELECT name FROM cards WHERE ban_tcg = 'Forbidden'", conn)
banned_list = set(banned['name'].tolist())

# Appliquer les deux filtres
exclude_list = staple_list | banned_list
key_cards_filtered = key_cards[~key_cards['card_name'].isin(exclude_list)]

print(f'Staples exclues  : {len(staple_list)} cartes')
print(f'Bannies exclues  : {len(banned_list)} cartes')
print(f'Cartes restantes : {len(key_cards_filtered)}')

## Part C — Score d'alerte combiné

In [ ]:
# views_week moyen des cartes clés filtrées par archetype
views_by_arch = key_cards_filtered.groupby('archetype').agg(
    avg_views_week=('views_week', 'mean'),
    max_views_week=('views_week', 'max'),
    n_key_cards=('card_name', 'count')
).reset_index()

# Fusion avec OCG meta_score
alert = agg[['archetype', 'meta_score_ocg', 'share_ocg', 'n_decks']].merge(
    views_by_arch, on='archetype', how='left'
)
alert['avg_views_week'] = alert['avg_views_week'].fillna(0)
alert['n_key_cards'] = alert['n_key_cards'].fillna(0)

# Score d'alerte
alert['alert_score'] = alert['meta_score_ocg'] * np.log1p(alert['avg_views_week'])
alert['alert_score_norm'] = (alert['alert_score'] / alert['alert_score'].max() * 100).round(1)

tcg_entry = (date.today() + relativedelta(months=LAG_MOIS)).strftime('%B %Y')
alert['tcg_entry_estimated'] = tcg_entry

alert_sorted = alert.sort_values('alert_score_norm', ascending=False)

print(f'Entrée TCG estimée : {tcg_entry}')
print()
print('TOP 20 — SCORE D\'ALERTE BOUTIQUES')
print('=' * 60)
print(alert_sorted[['archetype', 'alert_score_norm', 'meta_score_ocg', 'avg_views_week']].head(20).to_string(index=False))

## Part D — Top cartes à acheter par archetype

In [ ]:
top5 = alert_sorted['archetype'].head(5).tolist()

print('TOP CARTES À ACHETER PAR ARCHETYPE (staples + bannies exclues)')
print('=' * 65)

for arch in top5:
    score = alert_sorted[alert_sorted['archetype'] == arch]['alert_score_norm'].values[0]
    print(f'\n🃏 {arch} (alert_score: {score}/100) — entrée TCG estimée : {tcg_entry}')
    cards = key_cards_filtered[key_cards_filtered['archetype'] == arch].sort_values('views_week', ascending=False)
    cards = cards[['card_name', 'frequency', 'views_week']].head(8).copy()
    cards['frequency'] = (cards['frequency'] * 100).round(0).astype(int).astype(str) + '%'
    cards['views_week'] = cards['views_week'].astype(int)
    print(cards.to_string(index=False))

## Part E — Visualisation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

top15 = alert_sorted.head(15)
colors = ['#d62728' if s >= 70 else '#ff7f0e' if s >= 40 else '#2ca02c'
          for s in top15['alert_score_norm']]
ax1.barh(top15['archetype'][::-1], top15['alert_score_norm'][::-1], color=colors[::-1])
ax1.axvline(70, color='red', linestyle='--', alpha=0.5, label='Alerte haute (70)')
ax1.axvline(40, color='orange', linestyle='--', alpha=0.5, label='Alerte moyenne (40)')
ax1.set_xlabel('Score d\'alerte (0-100)')
ax1.set_title(f'Score alerte boutiques\n(entrée TCG : {tcg_entry})')
ax1.legend(fontsize=8)

plot_data = alert.dropna(subset=['avg_views_week']).head(30)
scatter = ax2.scatter(
    plot_data['meta_score_ocg'],
    np.log1p(plot_data['avg_views_week']),
    s=plot_data['alert_score_norm'] * 5 + 20,
    alpha=0.7, c=plot_data['alert_score_norm'], cmap='RdYlGn'
)
for _, row in plot_data.head(10).iterrows():
    ax2.annotate(row['archetype'], (row['meta_score_ocg'], np.log1p(row['avg_views_week'])),
                 fontsize=7, ha='left', va='bottom')
ax2.set_xlabel('Meta score OCG')
ax2.set_ylabel('log(views_week cartes core)')
ax2.set_title('Force OCG vs intérêt joueurs')
plt.colorbar(scatter, ax=ax2, label='Alert score')

plt.tight_layout()
out = os.path.join(os.getcwd(), '..', 'data', 'boutique_alert_score.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'Graphe sauvegardé : data/boutique_alert_score.png')

## Part F — Sauvegarde en DB

In [ ]:
# Score par archetype
alert_export = alert[['archetype', 'alert_score_norm', 'meta_score_ocg',
                       'share_ocg', 'avg_views_week', 'n_key_cards', 'tcg_entry_estimated']].copy()
alert_export.columns = ['archetype', 'alert_score', 'meta_score_ocg',
                        'share_ocg', 'avg_views_week', 'n_key_cards', 'tcg_entry_estimated']
alert_export['computed_at'] = datetime.now().strftime('%Y-%m-%d')
alert_export.to_sql('boutique_alerts', conn, if_exists='replace', index=False)

# Score par carte (filtrée)
card_alert = key_cards_filtered[key_cards_filtered['archetype'].isin(alert_sorted['archetype'].head(10))].copy()
card_alert = card_alert.merge(alert[['archetype', 'meta_score_ocg']], on='archetype')
card_alert['card_alert_score'] = (card_alert['meta_score_ocg'] * np.log1p(card_alert['views_week'])).round(3)
card_alert = card_alert[['archetype', 'card_name', 'frequency', 'views_week', 'card_alert_score']]
card_alert = card_alert.sort_values('card_alert_score', ascending=False)
card_alert.to_sql('boutique_card_alerts', conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print('Tables sauvegardées dans yugioh.db :')
print('  boutique_alerts      — score par archetype')
print('  boutique_card_alerts — score par carte (staples + bannies exclues)')
print()
print('Top 10 cartes à acheter :')
print(card_alert[['card_name', 'archetype', 'views_week', 'card_alert_score']].head(10).to_string(index=False))